In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from alerce.core import Alerce
from load import *

In [2]:
csv_dir = "csv_files"

print("=" * 50)
curves_data = load_curves(csv_dir)
print(f"Loaded {len(curves_data)} curves")

Processing curves: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12258/12258 [00:16<00:00, 744.24it/s]

Loaded 2869 curves


In [ ]:
import numpy as np
from scipy.signal import find_peaks

def is_periodic(t, mag, threshold=0.3):
    """
    FFT converts function into a set of frequencies (essentially a linear combination of sinusoidal functions).
    The idea is that if one frequency is extremely dominant, the original data is very periodic and therefore a 
    transient observation.
    """
    N = len(mag)
    
    mag_detrended = mag - np.mean(mag)
    
    window = np.hanning(N)
    mag_windowed = mag_detrended * window
    
    # FFT
    X = np.fft.fft(mag_windowed)
    freqs = np.fft.fftfreq(N, d=(t[1]-t[0]))  # sampling interval
    magnitude = np.abs(X)
    
    # Only positive frequencies
    pos_mask = freqs > 0
    freqs = freqs[pos_mask]
    magnitude = magnitude[pos_mask]
    
    # Find peaks
    peaks, props = find_peaks(magnitude, height=threshold * np.max(magnitude))
    
    if len(peaks) == 0:
        return False, None
    
    # Dominant frequency
    dom_freq = freqs[peaks[np.argmax(props['peak_heights'])]]
    dom_period = 1 / dom_freq
    
    return True, dom_period

In [ ]:
curves = load_curves(csv_dir)

for oid, t_norm, mag_norm, t, mag, ra, dec in curves:
    periodic, period = is_periodic(t_norm, mag_norm)
    if periodic:
        print(f"{oid} is periodic with period ≈ {period:.3f} (normalized time units)")
    else:
        print(f"{oid} is not periodic")